# Preparación y validación de datos

Proyecto 3 · Deep Learning y Sistemas Inteligentes · Sección 30

## Objetivo y criterio de éxito

Construir al menos 2,000 pares sobre 10 o más esquemas propios, separar la validación por esquema y aceptar únicamente consultas ejecutables en MySQL 8.4.10.

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "config.json").exists(), "Ejecuta el notebook desde la raíz o notebooks/."
ARTIFACTS = Path(os.environ.get("PROJECT3_ARTIFACTS", ROOT.parent / "artifacts_locales"))
SEED = 2026
print({"root": str(ROOT), "artifacts": str(ARTIFACTS), "seed": SEED})

{'root': 'C:\\Users\\Daniel\\Desktop\\Proyecto-3_Deep\\Proyecto-3_Grupo-1_DLYSI_Sec-30', 'artifacts': 'C:\\Users\\Daniel\\Desktop\\Proyecto-3_Deep\\artifacts_locales', 'seed': 2026}


## 1. Reconstrucción opcional

La entrega conserva los datos y manifiestos ya validados. Active estas banderas únicamente para reconstruirlos con Docker disponible.

In [2]:
RUN_DATA_BUILD = False
RUN_MYSQL_VALIDATION = False

if RUN_DATA_BUILD:
    subprocess.run([sys.executable, str(ROOT / 'scripts/generate_data_assets.py')], check=True)
else:
    print('Se usan los datos versionados; no se regeneraron archivos.')

if RUN_MYSQL_VALIDATION:
    subprocess.run([sys.executable, str(ROOT / 'scripts/validate_mysql.py')], check=True)
else:
    print('Se usa el manifiesto MySQL versionado; no se requiere Docker para leer este notebook.')

Se usan los datos versionados; no se regeneraron archivos.
Se usa el manifiesto MySQL versionado; no se requiere Docker para leer este notebook.


## 2. Volumen, motor y separación por esquema

In [3]:
manifest = json.loads((ROOT / 'resultados/dataset_manifest.json').read_text(encoding='utf-8'))
assert manifest['training_accepted'] >= 2000
assert manifest['training_schemas'] >= 10
assert manifest['validation_examples'] == 40
assert manifest['validation_schemas'] == 2
assert manifest['mysql_version'] == '8.4.10'
assert manifest['difficulty_validation'] == {'facil': 14, 'media': 14, 'dificil': 12}
{key: manifest[key] for key in ('training_candidates', 'training_accepted', 'training_discarded', 'training_schemas', 'validation_examples', 'validation_schemas', 'mysql_version', 'sql_mode')}

{'training_candidates': 2592,
 'training_accepted': 2592,
 'training_discarded': 0,
 'training_schemas': 12,
 'validation_examples': 40,
 'validation_schemas': 2,
 'mysql_version': '8.4.10',
 'sql_mode': 'ONLY_FULL_GROUP_BY,STRICT_TRANS_TABLES,NO_ZERO_IN_DATE,NO_ZERO_DATE,ERROR_FOR_DIVISION_BY_ZERO,NO_ENGINE_SUBSTITUTION'}

## 3. Revisión humana

In [4]:
import pandas as pd

review = pd.read_csv(ROOT / 'datos/revision_muestra.csv')
assert len(review) == 100
assert review['revisor'].notna().all()
assert manifest['human_review_complete'] is True
review.groupby(['revisor', 'decision']).size().rename('filas').reset_index()

,revisor,decision,filas
0,Pablo Daniel Barillas Moreno,aprobado_revision_humana,50
1,Wilson Alejandro Calderón Argueta,aprobado_revision_humana,50


## Resultado

El dataset quedó congelado con 2,592 pares aceptados, 0 descartes, 12 esquemas de entrenamiento, 2 esquemas inéditos y 100 filas confirmadas por revisión humana.